## Cross-Platform Phenotype Correlation — Incucyte vs Flow Cytometry (48h)

### Author: Eleni Aretaki

### Date: 2026-03-25

### Purpose

This notebook compares Incucyte growth phenotypes with flow cytometry–derived phenotypes
to assess cross-platform concordance across SWI/SNF knockout cell lines.

Correlations are computed using log2(KO/WT) ratios for matched treatments and cell lines.

### Analysis Overview

1. Load combined 48h flow cytometry dataset (final processed results)
2. Load Incucyte confluency dataset across multiple timepoints
3. Standardize treatment and cell line naming between platforms
4. Filter excluded compounds and cell lines
5. Merge datasets on:

   * cell line
   * treatment
6. Compute correlations between Incucyte and flow cytometry readouts
7. Generate regression plots for each Incucyte timepoint

### Flow Cytometry Readouts

* Apoptosis-positive fraction
* S-phase fraction

### Incucyte Readouts

Confluency-based KO/WT ratios at:

* 24h
* 48h
* 72h
* 96h

### Outputs

Two figure types are generated for each readout:

1. **Global regression plots**

   * All treatments combined
   * Pearson correlation shown

2. **Treatment-specific regression plots**

   * Separate regression per treatment
   * Shared axes

### Statistical Analysis

* Pearson correlation coefficient (r)
* Pearson p-value
* Spearman correlation coefficient (ρ)
* Spearman p-value

In [ ]:
# -------------------------------
# Imports
# -------------------------------
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import string
import os

In [ ]:
# -------------------------------
# Import data
# -------------------------------
fc = pd.read_csv(
"../fc_final_results_avg.csv"
)

inc = pd.read_csv(
"..incucyte_final_results_avg.txt",
sep="\t"
)

In [ ]:
# -------------------------------
# Preprocess data
# -------------------------------
# ------- Flow Cytometry --------
flow_cyt = fc.copy()

flow_cyt['Cell_Line'] = flow_cyt['Cell_Line'].str.replace(
    r'\s+KO$', '', regex=True
)

exclude_drugs = ['Aph']
flow_cyt = flow_cyt[~flow_cyt['Treatment'].isin(exclude_drugs)]

treatment_map = {
    'CPT': 'Camptothecin',
    'Ada': 'Adavosertib',  
    'MMS': 'Methyl-Methanesulfonate',
    'PBC': 'Palbociclib',
    'MLN': 'MLN4924',
    'HU': 'HydroxyUrea',
    'BIBR': 'BIBR1532',
    'COB': 'Cobimetinib',
    'LAP': 'Lapatinib',
    'PAC': 'Paclitaxel'
}

flow_cyt['Treatment'] = flow_cyt['Treatment'].map(treatment_map)

flow_cyt['Significance'] = flow_cyt['Significance'].fillna('')

flow_cyt_renamed = flow_cyt.rename(columns={
    'Cell_Line': 'cell_line',
    'Treatment': 'treatment',
    'Log2_KO_WT_Ratio': 'log2_KO_vs_WT',
    'Phenotype': 'readout'
})


# ------- Incucyte --------------
inc_data = inc.copy()

inc_data['cell_line_modification'] = inc_data[
'cell_line_modification'
].str.replace(r'_KO$', '', regex=True)

exclude_drugs = [
'BI-2536','Etoposide','Gemcitabine','IlludinS',
'Mirdametinib','NMS-873','Pyridostatin','Sapitinib','Topotecan'
]

exclude_cells = ['DPF2']

inc_data = inc_data[
~inc_data['treatment'].isin(exclude_drugs)
]

inc_data = inc_data[
~inc_data['cell_line_modification'].isin(exclude_cells)
]

inc_long = inc_data.melt(
    id_vars=['cell_line_modification','treatment'],
    value_vars=[
        'Confluency_after_24h',
        'Confluency_after_48h',
        'Confluency_after_72h',
        'Confluency_after_96h'
    ],
    var_name='timepoint',
    value_name='log2_KO_vs_WT'
)

inc_long['timepoint'] = inc_long['timepoint'].str.replace(
'Confluency_after_', '', regex=False
)

inc_long = inc_long.rename(columns={
    'cell_line_modification': 'cell_line'
})

In [ ]:
# -------------------------------
# Colors
# -------------------------------
palette = sns.color_palette("Paired", 12)

treatment_colors = {
    'Adavosertib': palette[1],
    'Camptothecin': palette[0],
    'HydroxyUrea': palette[2],
    'Methyl-Methanesulfonate': palette[3],
    'MLN4924': palette[4],
    'Palbociclib': palette[5],
    'Cobimetinib': palette[6],
    'Doxorubicin': palette[7],
    'Lapatinib': palette[8],
    'Niraparib': palette[9],
    'Paclitaxel': palette[10],
    'BIBR1532': palette[11],
}

In [ ]:
# -------------------------------
# Merge helper function
# -------------------------------
def merge_fc_incucyte(flow_cyt_renamed, inc_long, readout):

    fc_subset = flow_cyt_renamed[
        flow_cyt_renamed['readout'] == readout
    ]

    merged = pd.merge(
        inc_long,
        fc_subset,
        on=['treatment','cell_line'],
        how='inner',
        suffixes=('_inc','_fc')
    )

    return merged

In [ ]:
# -------------------------------
# Plotting functions
# -------------------------------
def plot_global_regression(merged_inc, output_dir, ylabel):

    os.makedirs(output_dir, exist_ok=True)
    plt.rcParams['font.family'] = 'Arial'

    timepoints = merged_inc['timepoint'].unique()

    for tp in timepoints:

        subset = merged_inc[merged_inc['timepoint'] == tp]

        pearson_r, pearson_p = pearsonr(
            subset['log2_KO_vs_WT_inc'],
            subset['log2_KO_vs_WT_fc']
        )

        spearman_r, spearman_p = spearmanr(
            subset['log2_KO_vs_WT_inc'],
            subset['log2_KO_vs_WT_fc']
        )

        plt.figure(figsize=(3.25,3))

        sns.scatterplot(
            data=subset,
            x='log2_KO_vs_WT_inc',
            y='log2_KO_vs_WT_fc',
            color='black',
            s=25,
            alpha=0.7
        )

        sns.regplot(
            data=subset,
            x='log2_KO_vs_WT_inc',
            y='log2_KO_vs_WT_fc',
            scatter=False,
            color='steelblue'
        )

        plt.text(
            0.15, 0.3,
            f'r = {pearson_r:.2f}\np = {pearson_p:.3g}',
            transform=plt.gca().transAxes,
            fontsize=8,
            color='steelblue',
            verticalalignment='top'
        )

        plt.xlabel(f"Incucyte log2(KO/WT) ({tp})")
        plt.ylabel(ylabel)
        plt.title(f"Confluency ({tp}) vs {ylabel.replace(' log2(KO/WT)','').replace(' KO-WT','').replace('FC ', '')}")

        plt.tight_layout()

        outfile = os.path.join(output_dir, f"Incucyte_{tp}_vs_FC.pdf")
        plt.savefig(outfile, bbox_inches='tight')
        plt.close()

        print(f"✅ Saved scatter plot: {outfile}")


def plot_treatment_regression(merged_inc, output_dir, ylabel):

    os.makedirs(output_dir, exist_ok=True)
    plt.rcParams['font.family'] = 'Arial'

    timepoints = merged_inc['timepoint'].unique()

    for tp in timepoints:

        subset = merged_inc[merged_inc['timepoint'] == tp].copy()

        subset = subset.replace([np.inf, -np.inf], np.nan).dropna(
            subset=['log2_KO_vs_WT_inc', 'log2_KO_vs_WT_fc']
        )

        pearson_r, pearson_p = pearsonr(
            subset['log2_KO_vs_WT_inc'],
            subset['log2_KO_vs_WT_fc']
        )

        fig, ax = plt.subplots(figsize=(5.25, 3.75))

        for treatment, color in treatment_colors.items():

            sub_trt = subset[subset['treatment'] == treatment]

            sns.regplot(
                data=sub_trt,
                x='log2_KO_vs_WT_inc',
                y='log2_KO_vs_WT_fc',
                ax=ax,
                scatter=True,
                ci=None,
                label=treatment,
                color=color,
                scatter_kws={'s': 25, 'alpha': 0.7}
            )

        ax.set_xlabel(f"Incucyte log2(KO/WT) ({tp})")
        ax.set_ylabel(ylabel)
        ax.set_title(f"Confluency ({tp}) vs {ylabel.replace(' log2(KO/WT)','').replace(' KO-WT','').replace('FC ', '')}")

        ax.text(
            0.5, 0.2,
            f'r = {pearson_r:.2f}\np = {pearson_p:.3g}',
            transform=ax.transAxes,
            ha='center', va='top',
            fontsize=8,
            color='steelblue'
        )

        ax.legend(
            title=None,
            fontsize=7,
            frameon=False,
            loc='center left',
            bbox_to_anchor=(1.02, 0.5)
        )

        plt.tight_layout()

        outfile = os.path.join(output_dir, f"Incucyte_{tp}_vs_FC.pdf")
        fig.savefig(outfile, bbox_inches="tight")
        plt.close(fig)

        print(f"✅ Saved figure: {outfile}")

In [ ]:
# -------------------------------
# MAIN Analysis
# -------------------------------

# ---------- APOPTOSIS ----------

merged_apoptosis = merge_fc_incucyte(
    flow_cyt_renamed,
    inc_long,
    readout="Apoptosis-Positive"
)

plot_global_regression(
    merged_apoptosis,
    output_dir="../Correlations/Apoptosis/Single_RegPlots",
    ylabel="FC Apoptosis KO-WT"
)

plot_treatment_regression(
    merged_apoptosis,
    output_dir="../Correlations/Apoptosis/Treatment_RegPlots",
    ylabel="FC Apoptosis KO-WT"
)


# ---------- S-PHASE ----------

merged_s = merge_fc_incucyte(
    flow_cyt_renamed,
    inc_long,
    readout="S-phase"
)

plot_global_regression(
    merged_s,
    output_dir="../Correlations/S-phase/Single_RegPlots",
    ylabel="FC S-phase log2(KO/WT)"
)

plot_treatment_regression(
    merged_s,
    output_dir="../Correlations/S-phase/Treatment_RegPlots",
    ylabel="FC S-phase log2(KO/WT)"
)